# Egyptian ALPR video pipeline — app-aligned notebook

This notebook runs the same three-stage inference stack used by the application:

1. COCO-pretrained YOLO11 vehicle detection.
2. The trained YOLO11 Master Plate detector.
3. The trained YOLO26 Egyptian character detector, including the app's crop enhancement, perspective correction, rotation variants, low-confidence retry, RTL ordering, and plate formatting.
4. The configured Keras/CRNN reader is used only when character detection does not produce text.

It does **not** install packages, download datasets, request API keys, or train replacement models. Run it from this repository with the same Python environment used by the app.


## 1. Locate the project and verify the runtime

The project root is detected whether Jupyter starts in the repository root or in `notebooks/`. No paths are tied to one machine.


In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import yaml
try:
    from IPython.display import Video, display
except (ImportError, ModuleNotFoundError):
    Video = None
    def display(value):
        print(value)


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'configs/model/two_stage.yaml').is_file() and (candidate / 'api').is_dir():
            return candidate
    raise RuntimeError('Could not find the AI-Tools-Project root. Start Jupyter inside the repository.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA is unavailable in this kernel. Select Python (AI Tools GPU - CUDA) '
        r'from D:\AI tools\venv\Scripts\python.exe, then restart and run all cells.'
    )
DEVICE: str | int = 0
print(f'Project: {PROJECT_ROOT}')
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda} | device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 2. Validate the datasets already used by the app

These are the repository datasets—not fresh Roboflow downloads. Validation resolves each `data.yaml`, verifies train/validation files, and reports the exact image and class counts.


In [ ]:
from scripts.train_detection import validate_training_setup

PLATE_DATA_YAML = PROJECT_ROOT / 'data/processed/Master_Plate_Dataset/data.yaml'
CHARACTER_DATA_YAML = PROJECT_ROOT / 'data/raw/dataset_Charcters_ready_plates/data.yaml'

dataset_rows = []
for stage, data_yaml in (('plate', PLATE_DATA_YAML), ('character', CHARACTER_DATA_YAML)):
    readiness = validate_training_setup(data_yaml, device='cpu')
    dataset_rows.append({'stage': stage, 'data_yaml': str(data_yaml), **readiness})

dataset_report = pd.DataFrame(dataset_rows).drop(columns=['device'])
display(dataset_report)


## 3. Resolve and load the same model stack as the app

Checkpoint selection mirrors `api.deps`: configured weights are used unless a newer completed run exists. The cascade itself is built by `build_two_stage_detector()` from `configs/model/two_stage.yaml`. The only notebook override is selecting CUDA when available.


In [ ]:
from api.deps import DET_WEIGHTS, get_reader
from src.detection.inference import build_two_stage_detector
from src.detection.workbench import discover_best_completed_run

CONFIG_PATH = PROJECT_ROOT / 'configs/model/two_stage.yaml'
with CONFIG_PATH.open(encoding='utf-8') as config_file:
    cascade_config = yaml.safe_load(config_file) or {}


def project_path(value: str | Path) -> Path:
    path = Path(value).expanduser()
    return path if path.is_absolute() else PROJECT_ROOT / path


def newest_checkpoint(configured: Path, runs_root: Path, fallback: Path | None = None) -> Path:
    selected = configured if configured.is_file() else fallback
    latest_run = discover_best_completed_run(runs_root)
    latest = latest_run / 'weights/best.pt' if latest_run else None
    if latest and latest.is_file():
        selected_mtime = selected.stat().st_mtime if selected and selected.is_file() else 0
        if latest.stat().st_mtime >= selected_mtime:
            selected = latest
    if selected is None or not selected.is_file():
        raise FileNotFoundError(f'No trained checkpoint found under {runs_root}')
    return selected.resolve()


plate_weights = newest_checkpoint(
    project_path(cascade_config['plate']['weights']),
    PROJECT_ROOT / 'models/detection',
    DET_WEIGHTS if DET_WEIGHTS.is_file() else None,
)
configured_character_weights = project_path(cascade_config['character']['weights'])
character_weights = (
    newest_checkpoint(configured_character_weights, PROJECT_ROOT / 'models/character')
    if cascade_config['character'].get('auto_select', True)
    else configured_character_weights.resolve()
)
if not character_weights.is_file():
    raise FileNotFoundError(f'Configured character checkpoint not found: {character_weights}')

cascade_config['plate']['weights'] = str(plate_weights)
cascade_config['character']['weights'] = str(character_weights)
cascade_config['character']['enabled'] = True
for stage in ('vehicle', 'plate', 'character'):
    cascade_config.setdefault(stage, {})['device'] = DEVICE

detector = build_two_stage_detector(cascade_config, PROJECT_ROOT)
reader = get_reader()

print(f'Plate checkpoint: {plate_weights}')
print(f'Character checkpoint: {character_weights}')
print(f'Fallback OCR: {type(reader).__name__ if reader is not None else "disabled"}')


## 4. App-equivalent annotation and recognition helpers

The API's clipping and drawing functions are reused directly. Character-detector text has priority; OCR is only a fallback, and all text passes through the app's Egyptian plate formatter.


In [ ]:
from api.routes.detection import MIN_COMPLETE_CHARACTER_READ, _clip_box, _draw_cascade
from src.postprocessing.plate_formatter import format_plate, validate_plate


def labels_for_frame(frame: np.ndarray, detections) -> tuple[list[str], list[dict]]:
    labels: list[str] = []
    records: list[dict] = []
    height, width = frame.shape[:2]
    for detection in detections:
        box = _clip_box(detection.plate.bbox, width, height)
        if box is None:
            labels.append('')
            continue
        x1, y1, x2, y2 = box
        crop = frame[y1:y2, x1:x2]
        character_text = getattr(detection, 'character_text', '')
        character_usable = bool(character_text and len(getattr(detection, 'characters', ())) >= MIN_COMPLETE_CHARACTER_READ)
        ocr_text = reader.read_plate(crop) if not character_usable and reader is not None else ''
        raw_text = character_text if character_usable else ocr_text
        formatted = format_plate(raw_text) if raw_text else ''
        if not character_usable and not validate_plate(formatted):
            formatted = ''
        labels.append(formatted)
        records.append({
            'plate_text': formatted,
            'text_source': 'character_detector' if character_usable else ('ocr' if ocr_text else None),
            'plate_confidence': round(detection.plate.confidence, 4),
            'vehicle_class': detection.vehicle.class_name,
            'vehicle_confidence': round(detection.vehicle.confidence, 4),
            'characters': len(getattr(detection, 'characters', ())),
        })
    return labels, records


## 5. Optional single-image smoke test

Set `TEST_IMAGE_PATH` to an image before running this cell. This is the quickest way to verify all three stages before processing a video.


In [ ]:
import matplotlib.pyplot as plt

TEST_IMAGE_PATH: Path | None = None  # Example: PROJECT_ROOT / 'data/sample.jpg'

if TEST_IMAGE_PATH is None:
    print('Set TEST_IMAGE_PATH to run the image smoke test.')
else:
    image = cv2.imread(str(TEST_IMAGE_PATH))
    if image is None:
        raise FileNotFoundError(f'Could not read {TEST_IMAGE_PATH}')
    started = time.perf_counter()
    image_detections = detector.predict(image)
    image_labels, image_records = labels_for_frame(image, image_detections)
    annotated = _draw_cascade(image, image_detections, image_labels)
    print(f'{len(image_detections)} plate(s) in {(time.perf_counter() - started) * 1000:.1f} ms')
    display(pd.DataFrame(image_records))
    plt.figure(figsize=(14, 8))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()


## 6. Process video with the same behavior as `/api/detect/video`

Detection runs every `FRAME_STRIDE` frames and the most recent result is drawn on intervening frames, matching the API. The output includes an annotated MP4, progress, and the same event-style summary used by the app.


In [ ]:
def process_video(
    input_path: Path,
    output_path: Path,
    confidence: float = 0.25,
    frame_stride: int = 3,
) -> dict:
    if not input_path.is_file():
        raise FileNotFoundError(f'Input video not found: {input_path}')
    if frame_stride < 1:
        raise ValueError('frame_stride must be at least 1')

    output_path.parent.mkdir(parents=True, exist_ok=True)
    detector.plate_detector.conf_threshold = confidence
    capture = cv2.VideoCapture(str(input_path))
    if not capture.isOpened():
        raise RuntimeError(f'Could not open video: {input_path}')

    fps = capture.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if width <= 0 or height <= 0:
        capture.release()
        raise RuntimeError('Video has invalid dimensions')

    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    if not writer.isOpened():
        capture.release()
        raise RuntimeError(f'Could not create output video: {output_path}')

    frame_index = 0
    frames_with_detections = 0
    total_detections = 0
    events: list[dict] = []
    current_detections = []
    current_labels: list[str] = []
    started = time.perf_counter()
    try:
        while True:
            ok, frame = capture.read()
            if not ok:
                break
            if frame_index % frame_stride == 0:
                current_detections = detector.predict(frame)
                current_labels, records = labels_for_frame(frame, current_detections)
                if current_detections:
                    frames_with_detections += 1
                    total_detections += len(current_detections)
                    if len(events) < 30:
                        events.append({
                            'frame': frame_index,
                            'time_seconds': round(frame_index / fps, 2),
                            'detections': len(current_detections),
                            'plates': [label for label in current_labels if label],
                            'records': records,
                        })
            writer.write(_draw_cascade(frame, current_detections, current_labels))
            frame_index += 1
            if frame_index % 100 == 0 or frame_index == total_frames:
                percent = 100 * frame_index / total_frames if total_frames else 0
                print(f'Processed {frame_index}/{total_frames or "?"} frames ({percent:.1f}%)')
    finally:
        capture.release()
        writer.release()

    return {
        'input': str(input_path),
        'output': str(output_path),
        'processed_frames': frame_index,
        'frames_with_detections': frames_with_detections,
        'total_detections': total_detections,
        'elapsed_seconds': round(time.perf_counter() - started, 2),
        'events': events,
    }


In [ ]:
INPUT_VIDEO = PROJECT_ROOT / 'input_video.mp4'  # Change this path.
OUTPUT_VIDEO = PROJECT_ROOT / 'reports/video_jobs/notebook_annotated.mp4'
CONFIDENCE = 0.25
FRAME_STRIDE = 3

video_summary = process_video(INPUT_VIDEO, OUTPUT_VIDEO, CONFIDENCE, FRAME_STRIDE)
display(pd.DataFrame([{key: value for key, value in video_summary.items() if key != 'events'}]))
event_rows = [
    {'frame': event['frame'], 'time_seconds': event['time_seconds'], 'detections': event['detections'], 'plates': ', '.join(event['plates'])}
    for event in video_summary['events']
]
display(pd.DataFrame(event_rows))
if Video is not None:
    display(Video(str(OUTPUT_VIDEO), embed=True, width=960))
else:
    print(f'Annotated video saved to: {OUTPUT_VIDEO}')


## Operational notes

- Restart the kernel after changing model files so no old model remains in memory.
- `FRAME_STRIDE=1` maximizes temporal coverage but is slower; the app defaults to `3`.
- The notebook deliberately does not retrain. Use the Training page or `scripts/train_detection.py` so training paths and memory safeguards remain centralized.
- If no vehicle is found, the configured cascade does not scan the full frame because `fallback_to_full_image` is disabled in the app configuration.
- Notebook outputs go under `reports/video_jobs/` and do not overwrite trained checkpoints or datasets.
